Import libraries and load API

In [32]:
# Import necessary libraries for working with APIs and environment variables
import json  # For parsing and working with JSON data
import os  # For accessing environment variables
from dotenv import load_dotenv  # For loading environment variables from .env file
from openai import OpenAI  # OpenAI API client for making requests to their models
import ollama  # Client for interacting with locally-run Ollama models
import time  # For measuring execution time and performance

# Load environment variables from .env file (contains API keys and secrets)
load_dotenv()

# Retrieve the OpenAI API key from environment variables
api_key = os.getenv("OPENAI_API_KEY")

# Initialize the OpenAI client with the API key
client = OpenAI(api_key=api_key)

# Define which OpenAI model to use for requests
MODEL = "gpt-4o-mini"

# Confirm that the client is ready to use
print("Client ready")

Client ready


1. Basic chat test and temperature effect on responses

In [ ]:
# Define the prompt/question to ask the model
# (Alternative: "Invent a useless product nobody would ever need. Explain why it exists.")
prompt = "Describe the world's strangest restaurant."

# List of temperature values to test
# Temperature controls randomness: 0 = deterministic, 2 = very random
temperatures = [0.1, 1.5]

# Loop through each temperature value
for temp in temperatures:

    # Print section header for clarity
    print("\n==============================")
    print(f"Temperature: {temp}")
    print("==============================\n")

    # Send a chat request to the Ollama model
    response = ollama.chat(
        model="tinyllama:1.1b",  # Use the TinyLlama model (lightweight, runs locally)
        messages=[
            {"role": "user", "content": prompt}  # Send the prompt as a user message
        ],
        options={
            "temperature": temp,  # Control randomness of the response
            "num_predict": 80  # Limit output to 80 tokens maximum
        }
    )

    # Extract and print the generated response
    print(response["message"]["content"])


Temperature: 0.1

The world's strangest restaurant is located in a remote, uninhabited island off the coast of the United States. It's called "The Restaurant at the End of the World," and it's been around for [...]
"
The restaurant has no windows or doors, and its walls are made entirely of glass. The ceiling is also made of

Temperature: 1.5

Surrounded by a lively and vibrant crowd, there was something about the restaurant that gave it an ethereal air of otherworldliness. The ambiance was electric with excitement and anticipation, [...]


2. Streaming chat - The model generates tokens progressively instead of waiting for the entire output to complete.

In [14]:
# Define the prompt to ask the model
prompt = "Explain what an LLM is in simple terms."

# Create a streaming response from OpenAI
# stream=True means we get tokens as they are generated, not all at once
stream = client.responses.create(
    model=MODEL,
    input=prompt,
    stream=True  # Enable streaming mode for real-time token generation
)

# Iterate through the stream of response events
for event in stream:

    # Check if this event is a text output delta (partial text)
    if event.type == "response.output_text.delta":
        # Print the text token without a newline (flush=True updates immediately)
        print(event.delta, end="", flush=True)

# Add a final newline after streaming completes
print()

An LLM, or Large Language Model, is a type of artificial intelligence that can understand and generate human language. Think of it like a very smart robot that has read a vast amount of text fr[...]


3. Testing system messages to control model behavior

In [ ]:
# Define the question to ask the model
question = "Explain artificial intelligence."

# ==========================================
# FIRST RESPONSE: University Professor Style
# ==========================================

print("University Professor------------------------")

# Request response in academic/professional tone
response_1 = client.responses.create(
    model=MODEL,
    # instructions acts as a "system message" that guides the model's behavior
    instructions="You are a university professor explaining concepts formally.",
    input=question,
    max_output_tokens=50  # Limit output length
)

# Extract and print the response
print(response_1.output_text)

# ==========================================
# SECOND RESPONSE: Explain to a Child Style
# ==========================================

print("Explain to a Child----------------------")

# Request response in simple, kid-friendly language
response_2 = client.responses.create(
    model=MODEL,
    # Different instruction to change the tone and complexity
    instructions="Explain concepts like you are talking to a 10-year-old child.",
    input=question,
    max_output_tokens=50  # Limit output length
)

# Extract and print the response
print(response_2.output_text)


University Professor

Artificial intelligence (AI) refers to the simulation of human intelligence processes by machines, particularly computer systems. This field encompasses a variety of techniques and methodologie[...]
"
Explain to a Child

Okay! Imagine you have a super-smart robot friend. This robot can listen, learn, and solve problems just like we do, but it uses a lot of information and special rules to help it think.

Artificial intelligence, or AI for short, is


4. Testing with tools (function calling)

In [18]:
# Import json (already imported above, but including for clarity)
import json

# Sample text containing contact information that we want the model to extract
text = """
Hello team,

Please contact john.doe@company.com for technical support.
You can also reach maria.garcia@gmail.com for marketing questions.

For urgent matters:
support@openai.com

Thank you.
"""

# ===========================================
# DEFINE THE TOOL/FUNCTION SCHEMA
# ===========================================
# Tools tell the model what functions it can call and what data they expect

tools = [
    {
        "type": "function",
        "name": "extract_contact_information",  # Function name
        "description": "Extract emails and identify their purpose from a text.",  # What it does
        "parameters": {
            "type": "object",  # Parameters are structured as a JSON object
            "properties": {  # Define each property/field

                "contacts": {
                    "type": "array",  # This is a list
                    "description": "List of extracted contacts",
                    "items": {
                        "type": "object",  # Each item in the list is an object
                        "properties": {

                            "email": {
                                "type": "string",  # Email is text
                                "description": "Detected email address"
                            },

                            "purpose": {
                                "type": "string",  # Purpose is text
                                "description": "Reason or department associated with the email"
                            }

                        },
                        "required": ["email", "purpose"]  # Both fields are required
                    }
                },

                "total_emails": {
                    "type": "integer",  # Total is a whole number
                    "description": "Total number of emails found"
                }

            },
            "required": ["contacts", "total_emails"]  # Both fields are required in the response
        }
    }
]

# ===========================================
# CALL THE MODEL WITH TOOLS
# ===========================================

response = client.responses.create(
    model=MODEL,
    input=f"""
Extract all contact information from this text.
For each email, identify its purpose.

Text:
{text}
""",
    tools=tools  # Pass the tool definitions to the model
)

# ===========================================
# EXTRACT AND DISPLAY TOOL OUTPUT
# ===========================================

# Get the first output (which contains the tool call)
tool_call = response.output[0]

# Parse the JSON arguments that the model prepared for the function
arguments = json.loads(tool_call.arguments)

# Display the results
print("\n==============================")
print("Extracted Contact Information")
print("==============================\n")

# Loop through each contact and display its information
for contact in arguments["contacts"]:
    print(f"Email: {contact['email']}")
    print(f"Purpose: {contact['purpose']}")
    print()

# Display the total count
print(f"Total emails found: {arguments['total_emails']}")


Extracted Contact Information

Email: john.doe@company.com
Purpose: technical support

Email: maria.garcia@gmail.com
Purpose: marketing questions

Email: support@openai.com
Purpose: urgent matters

Total emails found: 3


In [33]:
# ===========================================
# LOAD API KEY AND INITIALIZE CLIENT
# ===========================================

# Load environment variables from .env file
load_dotenv()

# Get the OpenAI API key from environment
api_key = os.getenv("OPENAI_API_KEY")

# Create an OpenAI client instance
client = OpenAI(api_key=api_key)

# ===========================================
# SETUP BENCHMARK TEST
# ===========================================

# Define the prompt we'll use to test all models
prompt = """
Explain what Retrieval-Augmented Generation (RAG) is,
how embeddings work,
and why vector databases are useful.
Keep the answer concise.
"""

# List of OpenAI models to test
openai_models = [
    "gpt-5-nano",      # Smallest/fastest OpenAI model
    "gpt-4.1-nano",    # Medium OpenAI model
    "gpt-4o-mini"      # More capable OpenAI model
]

# List of Ollama models to test (run locally)
ollama_models = [
    "tinyllama:1.1b",  # Very small local model
    "llama3.2:3b"      # Larger local model
]

# ===========================================
# OPENAI MODEL BENCHMARK
# ===========================================

print("\n========================================")
print("OPENAI MODEL PERFORMANCE")
print("========================================\n")

# Loop through each OpenAI model
for model in openai_models:

    print(f"\nTesting model: {model}")
    print("-" * 40)

    # Record the start time
    start = time.perf_counter()

    # Make a request to the OpenAI model
    response = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=120  # Generate up to 120 tokens
    )

    # Record the end time
    end = time.perf_counter()

    # Calculate how long the request took
    latency = end - start

    # Extract the generated text from the response
    output = response.output_text

    # Calculate metrics
    characters = len(output)  # Total characters in the response
    words = len(output.split())  # Total words in the response

    # Calculate generation speed (characters per second)
    chars_per_second = characters / latency
    # Calculate words per second (alternative metric)
    words_per_second = words / latency

    # Display results
    print(f"Latency: {latency:.2f} seconds")
    print(f"Characters/sec: {chars_per_second:.2f}")

    print("\nResponse Preview:\n")
    # Show first 300 characters of the response
    print(output[:300])

# ===========================================
# OLLAMA MODEL BENCHMARK
# ===========================================

print("\n\n========================================")
print("OLLAMA MODEL PERFORMANCE")
print("========================================\n")

# Loop through each Ollama model
for model in ollama_models:

    print(f"\nTesting model: {model}")
    print("-" * 40)

    # Record the start time
    start = time.perf_counter()

    # Make a request to the Ollama model
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "num_predict": 120  # Generate up to 120 tokens
        }
    )

    # Record the end time
    end = time.perf_counter()

    # Calculate how long the request took
    latency = end - start

    # Extract the generated text from the response
    output = response["message"]["content"]

    # Calculate metrics
    characters = len(output)  # Total characters in the response
    words = len(output.split())  # Total words in the response

    # Calculate generation speed (characters per second)
    chars_per_second = characters / latency
    # Calculate words per second (alternative metric)
    words_per_second = words / latency

    # Display results
    print(f"Latency: {latency:.2f} seconds")
    print(f"Characters/sec: {chars_per_second:.2f}")

    print("\nResponse Preview:\n")
    # Show first 300 characters of the response
    print(output[:300])


OPENAI MODEL PERFORMANCE


Testing model: gpt-5-nano
----------------------------------------
Latency: 1.53 seconds
Characters/sec: 0.00

Response Preview:



Testing model: gpt-4.1-nano
----------------------------------------
Latency: 1.87 seconds
Characters/sec: 279.81

Response Preview:

Retrieval-Augmented Generation (RAG) combines a language model with an external knowledge base by retrieving relevant documents or data using embeddings and vector databases. Embeddings are den[...]
"
Testing model: gpt-4o-mini
----------------------------------------
Latency: 2.45 seconds
Characters/sec: 288.27
Response Preview:

**Retrieval-Augmented Generation (RAG)** is a model architecture that combines information retrieval and text generation. It retrieves relevant documents from a knowledge base or corpus and use[...]
"

OLLAMA MODEL PERFORMANCE


Testing model: tinyllama:1.1b
----------------------------------------
Latency: 3.21 seconds
Characters/sec: 168.52

Response Preview:

Retrieva